## Test X.ai API (GROQ)

In [1]:
import os
from openai import OpenAI
import openai

openai.api_key = os.getenv("XAI_API_KEY")

client = OpenAI(
  api_key=openai.api_key ,
  base_url="https://api.x.ai/v1",
)

completion = client.chat.completions.create(
  model="grok-3-latest",
  messages=[
    {"role": "system", "content": "You are a PhD-level mathematician."},
    {"role": "user", "content": "What is 2 + 2?"},
  ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='2 + 2 equals 4.\n\nThis is a basic arithmetic operation where two numbers, each of value 2, are combined to produce a sum of 4. In the context of natural numbers or integers, this result is fundamental and can be understood through counting (e.g., combining two groups of two items results in four items) or through the properties of addition in the number system.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [4]:
from git import Repo
import tempfile
import os

# URL of the repo you want to ingest
REPO_URL = "https://github.com/RWTH-EBC/AixLib.git"

# Clone into a temp folder
tmp_dir = tempfile.mkdtemp()
Repo.clone_from(REPO_URL, tmp_dir)
print(f"Cloned into {tmp_dir}")

Cloned into /var/folders/8w/nhzjf7wn3f7bxb6vlmbqbwfc0000gn/T/tmp4szhn_ub


In [5]:
import glob
import os

# 読み込み対象の拡張子
EXTENSIONS = [".md", ".py", ".txt", ".rst"]

def load_repo_texts(root_dir):
    docs = []
    for ext in EXTENSIONS:
        pattern = os.path.join(root_dir, "**", f"*{ext}")
        for path in glob.glob(pattern, recursive=True):
            # ファイルでなければスキップ
            if not os.path.isfile(path):
                continue
            # サイズが大きすぎるものはスキップ
            if os.path.getsize(path) > 1e6:
                continue
            try:
                with open(path, encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                docs.append({
                    "path": os.path.relpath(path, root_dir),
                    "content": text
                })
            except Exception as e:
                # 万が一の読み込みエラーも無視
                print(f"Warning: failed to read {path}: {e}")
    return docs

# 使い方
documents = load_repo_texts(tmp_dir)
print(f"Loaded {len(documents)} files")

Loaded 1023 files


In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = []
for doc in documents:
    texts = splitter.split_text(doc["content"])
    for i, txt in enumerate(texts):
        chunks.append({
            "id": f"{doc['path']}-{i}",
            "text": txt
        })

print(f"Created {len(chunks)} text chunks")

Created 23751 text chunks


In [10]:
import openai
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

openai.api_key = os.getenv("XAI_API_KEY")
embedder = OpenAIEmbeddings()

# texts: list of strings
texts = [c["text"] for c in chunks]
metadatas = [{"source": c["id"]} for c in chunks]

# Create FAISS index
index = FAISS.from_texts(texts, embedder, metadatas=metadatas)

In [11]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# ↓ すでに保存してある faiss_index フォルダを読み込む
embedder = OpenAIEmbeddings()
index = FAISS.load_local(
    "faiss_index",
    embedder,
    allow_dangerous_deserialization=True  # <-- これを追加
)

In [16]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# LLM
llm = ChatOpenAI(model_name="grok-3-beta",   
                 api_key=openai.api_key ,
                 base_url="https://api.x.ai/v1",
                 temperature=0)

# RetrievalQA
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",         # or "map_reduce", "refine", etc.
    retriever=index.as_retriever(),
    return_source_documents=True
)

# Ask a question
query = "Where is heat exchanger models in this project?"
result = qa(query)

print("Answer:\n", result["result"])
print("\nSource Chunks:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

Answer:
 The heat exchanger models in this project are located under the following paths within the `AixLib` library:

- **General Heat Exchanger Models**:
  - `AixLib.Fluid.HeatExchangers.SensibleCooler_T`
  - `AixLib.Fluid.HeatExchangers.WetCoilEffectivenessNTU`

- **Active Beams (Cooling and Heating)**:
  - `AixLib.Fluid.HeatExchangers.ActiveBeams.Cooling`
  - `AixLib.Fluid.HeatExchangers.ActiveBeams.CoolingAndHeating`
  - Related base classes and data are under `AixLib.Fluid.HeatExchangers.ActiveBeams.BaseClasses` and `AixLib.Fluid.HeatExchangers.ActiveBeams.Data`.

- **Radiators**:
  - `AixLib.Fluid.HeatExchangers.Radiators.RadiatorEN442_2`

- **Examples and Validation Models**:
  - Examples are located in `AixLib.Fluid.HeatExchangers.Examples`, including models like `DryCoilEffectivenessNTUPControl`, `WaterCooler_T`, `WaterHeater_T`, and others.
  - Validation models are under `AixLib.Fluid.HeatExchangers.Validation`, including models like `ConstantEffectiveness`, `DryCoilEffecti

In [17]:
# Ask a question
query = "Construct room air conditioner model by AixLib."
result = qa(query)

print("Answer:\n", result["result"])
print("\nSource Chunks:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

Answer:
 To construct a room air conditioner model using **AixLib**, you can leverage the library's components for HVAC systems, particularly focusing on models related to compressors, heat exchangers, and air handling. AixLib, developed at RWTH Aachen University's E.ON Energy Research Center, provides a comprehensive set of models for building performance simulations, including HVAC systems. Below, I will outline a general approach to constructing a room air conditioner model using components from AixLib. Since the provided context does not include a specific pre-built air conditioner model, we will build one using relevant sub-components.

### Step-by-Step Guide to Construct a Room Air Conditioner Model in AixLib

1. **Understand the Components of a Room Air Conditioner**:
   A typical room air conditioner (split or window unit) operates on a vapor-compression refrigeration cycle and includes:
   - A compressor (to compress the refrigerant).
   - An evaporator (to cool the room air).